In [2]:
import os
if os.path.basename(os.getcwd()) == "inspeccion":
    os.chdir("..")

import pandas as pd
from utils.funciones_filtrado import tipo_nulo_unicos_x_columna

In [3]:
base_inventario = pd.read_excel('inventario.xlsx')
base_inventario.head()

,producto_id,producto,proveedor,stock_actual,stock_minimo,costo,ultima_compra
0,P001,Americano,Café Veracruz SA,120.0,30,8.5,2024-11-01
1,P002,Latte,Café Veracruz SA,95.0,25,9,2024-10-15
2,P003,Cappuccino,Café Veracruz SA,NaN,20,8.75,2024-09-20
3,P004,Espresso,Café Veracruz SA,200.0,50,6,2024-11-10
4,P005,frappe chocolate,Bebidas Frías MX,60.0,15,12,2024-10-01


In [3]:
no_registros, no_columnas = base_inventario.shape
print(f"Número de productos registrados: {no_registros}, número de columnas de la tabla: {no_columnas}")

Número de productos registrados: 17, número de columnas de la tabla: 7


Notamos que a primera vista en la columna 'stock_actual' tenemos un valor nulo, las demás parecen no tener algún tipo de error en sus primero registros.

Pero al ver las primeras 2 columnas me recuerda que la tabla 'productos' también tiene esas mismas columnas, habrá que ver si tienen relación, porque si no la tienen, propongo hacer un cambio de nombre para mejorar la comprensión de las tablas y facilitar los cruces.

In [4]:
inspeccion = tipo_nulo_unicos_x_columna(base_inventario)
inspeccion

,columna,tipo de dato,cantidad de valores nulos,cantidad de valores unicos
0,producto_id,object,1,16
1,producto,object,0,16
2,proveedor,object,0,7
3,stock_actual,float64,1,15
4,stock_minimo,int64,0,10
5,costo,object,0,14
6,ultima_compra,object,0,12


Observamos que la columna 'stock_actual' tiene un tipo float64, que es probablemente que es por el valor nulo que observamos en la tercera fila o es que si realmente exista un valor flotante, pero habrá que investigar la logica del invetario para ver si es posible que exista fracciones de stock.

También notamos que la columna 'costo' no es un tipo numerico, es decir, existe algún registro que no sea nulo, entero o flotante, por lo que habrá que insperccionar esa columna. Además la columna 'ultima_compra' no es del tipo de fecha.

Notamos que hay tanto en 'producto_id' como en 'stock_actual' hay valores faltantes, lo cual es malo al momento de hacer cruces con tablas además de que desconocemos un valor del stock, por lo que no sabemos si nos quedaremos sin ese producto.

Además vemos que en la columna 'producto' sólo hay 16 productos, es decir, hay un prodcuto que se repite porque el numero de registros es de 17 y esto es curioso ya que hay que ver si el mismo producto vienen de proovedores diferentes y si tenemos el mismo stock para ese producto.

In [5]:
#inspección a la columna costo, del porque esa columna no es numerica

no_numero = base_inventario[pd.to_numeric(base_inventario['costo'], errors='coerce').isna()]
no_numero

,producto_id,producto,proveedor,stock_actual,stock_minimo,costo,ultima_compra
16,P016,Americano,Café Veracruz SA,45.0,30,ocho pesos,2024-07-01


Al descartar todos los registros que no se convirtieron en número en la columna costo, se puede observar que hay un registro que en lugar de estar escrito con número se escribió numericamente

In [6]:
producto_repetido = base_inventario[base_inventario['producto'].duplicated(keep=False)]
producto_repetido

,producto_id,producto,proveedor,stock_actual,stock_minimo,costo,ultima_compra
0,P001,Americano,Café Veracruz SA,120.0,30,8.5,2024-11-01
16,P016,Americano,Café Veracruz SA,45.0,30,ocho pesos,2024-07-01


En la columna de producto, efectivamente hay un producto repetido, que viene del mismo proveedor, no coinciden en la cantidad de su stock actual, ni tienen el mismo id

In [4]:
base_inventario[base_inventario['producto_id'].isna()]

,producto_id,producto,proveedor,stock_actual,stock_minimo,costo,ultima_compra
14,NaN,Vaso Desechable 16oz,Empaques MX,500.0,100,1.2,2024-10-15


Investigamos cuál es el producto que no tiene un id

In [7]:
base_inventario['proveedor'].unique()

array(['Café Veracruz SA', 'Bebidas Frías MX', 'Infusiones del Sur',
       'Panadería Local', 'Lácteos Puebla', 'Proveedores Generales',
       'Empaques MX'], dtype=object)

Verificamos que ningún proveedor este mal escrito, y así evitar generar duplicidad.